> ## SOLUTION / ANSWER KEY &mdash; Lab 8.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-the-supervisor.ipynb`](../lab-02-the-supervisor.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.2 &mdash; The Supervisor (Router)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Route a message to the specialist(s) whose scope it matches
- Handle a message with TWO intents -> route to both
- Fall back to a general agent when nothing matches

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The supervisor (router) pattern](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-02"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The **supervisor** (router) is the backbone of a multi-agent system (deck slide 6): it receives the
message, decides **which specialist** should handle it, and routes there. It is exactly Module 7's **route**
pattern &mdash; but the destinations are **agents**, not code branches. A message can carry **two intents**
(a billing problem *and* a crash), so routing returns a **list**; and an unmatched message falls back to a
**general** agent (the escape hatch). This classification is genuine rule-based plumbing &mdash; no LLM
needed to decide who owns a ticket.

In [ ]:
# A tiny Specialist just knows its name + the keywords that put a message in its scope.
print("we will route to whichever specialists a message is in scope for -- possibly several")

## Build it
Complete `route`: collect every specialist whose scope matches, and fall back to `["general"]`.

In [ ]:
class Specialist:
    def __init__(self, name, keywords): self.name = name; self.keywords = keywords
    def in_scope(self, message):
        return any(k in message.lower() for k in self.keywords)

billing = Specialist("billing", ["charge", "refund", "invoice", "billed"])
tech    = Specialist("tech", ["crash", "error", "login", "bug", "broken"])
SPECIALISTS = [billing, tech]

def route(message, specialists):
    hits = [s.name for s in specialists if s.in_scope(message)]
    return hits if hits else ["general"]

try:
    print("charge msg :", route("I was charged twice", SPECIALISTS))
    print("crash msg  :", route("the app keeps crashing", SPECIALISTS))
    print("both       :", route("charged twice and the app crashes", SPECIALISTS))
    print("unknown    :", route("what are your hours?", SPECIALISTS))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A billing message routes to `["billing"]`, a crash to `["tech"]`, and a **two-intent** message to **both** &mdash; routing returns a *list*.
- An unmatched message falls back to `["general"]` &mdash; the router never drops a ticket on the floor.
- In Lab 8.11 these names become the keys into a dict of **real** specialist agents; here you just decide *who*.

## Your turn (open task &mdash; no grader)
Add a **third** specialist (e.g. `account` with keywords like `"password"`, `"email"`, `"cancel"`) to
`SPECIALISTS` and re-run over a message that hits it. **What good looks like:** the router now sends
account-related tickets to `account`, two-intent messages reach both owners, and anything unmatched still
falls back to `general`. That is the supervisor whose destinations you'll wire to real agents later.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
account = Specialist("account", ["password", "email", "cancel"])
SPECIALISTS3 = [billing, tech, account]
print("account :", route("please cancel my account", SPECIALISTS3))
print("two     :", route("charged twice and I forgot my password", SPECIALISTS3))
print("unknown :", route("what are your hours?", SPECIALISTS3))

---
### Key takeaway
The supervisor is Module 7's route pattern -- but routing to AGENTS. Returning a list lets one message reach several specialists; the general fallback keeps it safe. Next: how those agents share what they find.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>